In [2]:
# ============================================
# CELL 1: IMPORT & LOAD CLEANED DATA
# ============================================
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

PROCESSED_PATH = Path(r'D:\CAPSTONE\data\processed')
FINAL_PATH = Path(r'D:\CAPSTONE\data\final')

# Load data yang sudah bersih
orders = pd.read_csv(PROCESSED_PATH / 'orders_clean.csv', parse_dates=[
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

order_items = pd.read_csv(PROCESSED_PATH / 'order_items_clean.csv')
products = pd.read_csv(PROCESSED_PATH / 'products_clean.csv')
customers = pd.read_csv(PROCESSED_PATH / 'customers_clean.csv')
reviews = pd.read_csv(PROCESSED_PATH / 'reviews_clean.csv')

print("✅ Semua cleaned data berhasil di-load!")
print(f"   orders      : {orders.shape}")
print(f"   order_items : {order_items.shape}")
print(f"   products    : {products.shape}")
print(f"   customers   : {customers.shape}")
print(f"   reviews     : {reviews.shape}")

✅ Semua cleaned data berhasil di-load!
   orders      : (96470, 8)
   order_items : (112650, 7)
   products    : (32951, 10)
   customers   : (99441, 5)
   reviews     : (98673, 8)


In [3]:
# ============================================
# CELL 2: DELIVERY & TIME FEATURES
# ============================================
print("⚙️ Membuat delivery & time features...\n")

orders_feat = orders.copy()

# --- Feature 1: delivery_time (hari) ---
# Berapa hari dari order sampai barang diterima customer
orders_feat['delivery_time_days'] = (
    orders_feat['order_delivered_customer_date'] -
    orders_feat['order_purchase_timestamp']
).dt.days

# --- Feature 2: delivery_delay (hari) ---
# Positif = terlambat dari estimasi
# Negatif = lebih cepat dari estimasi
orders_feat['delivery_delay_days'] = (
    orders_feat['order_delivered_customer_date'] -
    orders_feat['order_estimated_delivery_date']
).dt.days

# --- Feature 3: is_late_delivery ---
# Boolean: apakah pengiriman terlambat?
orders_feat['is_late_delivery'] = (
    orders_feat['delivery_delay_days'] > 0
).astype(int)

# --- Feature 4: order_month, order_year, order_dayofweek ---
orders_feat['order_month'] = orders_feat['order_purchase_timestamp'].dt.month
orders_feat['order_year'] = orders_feat['order_purchase_timestamp'].dt.year
orders_feat['order_dayofweek'] = orders_feat['order_purchase_timestamp'].dt.dayofweek
# 0=Monday, 6=Sunday

# Verifikasi
print("✅ Features yang dibuat:")
new_features = [
    'delivery_time_days', 'delivery_delay_days',
    'is_late_delivery', 'order_month',
    'order_year', 'order_dayofweek'
]
for f in new_features:
    print(f"   {f}: {orders_feat[f].dtype} | "
          f"min={orders_feat[f].min():.0f}, "
          f"max={orders_feat[f].max():.0f}, "
          f"mean={orders_feat[f].mean():.1f}")

late_pct = orders_feat['is_late_delivery'].mean() * 100
print(f"\n📊 Insight awal:")
print(f"   {late_pct:.1f}% pengiriman terlambat dari estimasi")
print(f"   Rata-rata waktu pengiriman: {orders_feat['delivery_time_days'].mean():.1f} hari")

⚙️ Membuat delivery & time features...

✅ Features yang dibuat:
   delivery_time_days: int64 | min=0, max=209, mean=12.1
   delivery_delay_days: int64 | min=-147, max=188, mean=-11.9
   is_late_delivery: int64 | min=0, max=1, mean=0.1
   order_month: int32 | min=1, max=12, mean=6.0
   order_year: int32 | min=2016, max=2018, mean=2017.5
   order_dayofweek: int32 | min=0, max=6, mean=2.8

📊 Insight awal:
   6.8% pengiriman terlambat dari estimasi
   Rata-rata waktu pengiriman: 12.1 hari


In [4]:
# ============================================
# CELL 3: ORDER VALUE FEATURES
# ============================================
print("⚙️ Membuat order value features...\n")

# Hitung total per order dari order_items
order_value = order_items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    item_count=('order_item_id', 'count')
).reset_index()

# Tambahkan total_order_value (harga + ongkir)
order_value['total_order_value'] = (
    order_value['total_price'] +
    order_value['total_freight']
)

print("✅ Features order value:")
print(f"   total_price       : harga produk per order")
print(f"   total_freight     : ongkir per order")
print(f"   item_count        : jumlah item per order")
print(f"   total_order_value : total bayar customer")

print(f"\n📊 Statistik order value:")
print(order_value[['total_price','total_freight','total_order_value']].describe().round(2))

⚙️ Membuat order value features...

✅ Features order value:
   total_price       : harga produk per order
   total_freight     : ongkir per order
   item_count        : jumlah item per order
   total_order_value : total bayar customer

📊 Statistik order value:
       total_price  total_freight  total_order_value
count     98666.00       98666.00           98666.00
mean        137.75          22.82             160.58
std         210.65          21.65             220.47
min           0.85           0.00               9.59
25%          45.90          13.85              61.98
50%          86.90          17.17             105.29
75%         149.90          24.04             176.87
max       13440.00        1794.96           13664.08


In [5]:
# ============================================
# CELL 4: RFM TABLE — INPUT UNTUK AI SEGMENTATION
# ============================================
print("⚙️ Membuat RFM Table (input untuk Customer Segmentation AI)...\n")

# --- Step 1: Join orders + customers ---
orders_customers = orders_feat.merge(
    customers[['customer_id', 'customer_unique_id',
               'customer_city', 'customer_state']],
    on='customer_id',
    how='left'
)

# --- Step 2: Join dengan order value ---
orders_full = orders_customers.merge(
    order_value,
    on='order_id',
    how='left'
)

print(f"✅ Join orders + customers + order_value: {orders_full.shape}")

# --- Step 3: Tentukan tanggal referensi ---
# Tanggal paling akhir dalam dataset + 1 hari
reference_date = orders_full['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f"   Reference date untuk Recency: {reference_date.date()}")

# --- Step 4: Hitung RFM per customer_unique_id ---
rfm = orders_full.groupby('customer_unique_id').agg(
    # RECENCY: berapa hari sejak terakhir beli
    recency=('order_purchase_timestamp',
             lambda x: (reference_date - x.max()).days),
    # FREQUENCY: berapa kali beli
    frequency=('order_id', 'count'),
    # MONETARY: total uang yang dikeluarkan
    monetary=('total_order_value', 'sum')
).reset_index()

print(f"\n✅ RFM Table berhasil dibuat!")
print(f"   Shape: {rfm.shape}")
print(f"\n📊 Statistik RFM:")
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2))

print(f"\n🔍 Preview RFM Table:")
print(rfm.head(5).to_string(index=False))

⚙️ Membuat RFM Table (input untuk Customer Segmentation AI)...

✅ Join orders + customers + order_value: (96470, 21)
   Reference date untuk Recency: 2018-08-30

✅ RFM Table berhasil dibuat!
   Shape: (93350, 4)

📊 Statistik RFM:
        recency  frequency  monetary
count  93350.00   93350.00  93350.00
mean     237.95       1.03    165.17
std      152.59       0.21    226.30
min        1.00       1.00      9.59
25%      114.00       1.00     63.01
50%      219.00       1.00    107.78
75%      346.00       1.00    182.50
max      714.00      15.00  13664.08

🔍 Preview RFM Table:
              customer_unique_id  recency  frequency  monetary
0000366f3b9a7992bf8c76cfdf3221e2      112          1    141.90
0000b849f77a49e4a4ce2b2a4ca5be3f      115          1     27.19
0000f46a3911fa3c0805444483337064      537          1     86.22
0000f6ccb0745a6a4b88665a16c9f078      321          1     43.62
0004aac84e0df4da2b147fca70cf8255      288          1    196.89


In [6]:
# ============================================
# CELL 5: MASTER DATASET (untuk BI Dashboard)
# ============================================
print("⚙️ Membuat Master Dataset...\n")

# Join semua tabel
master = orders_full.copy()

# Join dengan reviews
master = master.merge(
    reviews[['order_id', 'review_score', 'sentiment_label',
             'review_comment_message']],
    on='order_id',
    how='left'
)

# Join dengan products melalui order_items
order_items_products = order_items.merge(
    products[['product_id', 'product_category_name',
              'product_category_english']],
    on='product_id',
    how='left'
)

# Ambil 1 produk per order (produk pertama/utama)
main_product = order_items_products.groupby('order_id').first().reset_index()

master = master.merge(
    main_product[['order_id', 'product_id',
                  'product_category_english', 'price']],
    on='order_id',
    how='left'
)

# Pilih kolom yang relevan untuk dashboard
cols_to_keep = [
    'order_id', 'customer_unique_id',
    'customer_city', 'customer_state',
    'order_purchase_timestamp', 'order_year', 'order_month',
    'order_dayofweek', 'delivery_time_days',
    'delivery_delay_days', 'is_late_delivery',
    'total_price', 'total_freight', 'total_order_value',
    'item_count', 'product_category_english',
    'review_score', 'sentiment_label'
]

master_final = master[cols_to_keep].copy()

print(f"✅ Master Dataset berhasil dibuat!")
print(f"   Shape: {master_final.shape}")
print(f"   Kolom: {master_final.columns.tolist()}")
print(f"\n🔍 Preview:")
print(master_final.head(3).to_string())

⚙️ Membuat Master Dataset...

✅ Master Dataset berhasil dibuat!
   Shape: (96470, 18)
   Kolom: ['order_id', 'customer_unique_id', 'customer_city', 'customer_state', 'order_purchase_timestamp', 'order_year', 'order_month', 'order_dayofweek', 'delivery_time_days', 'delivery_delay_days', 'is_late_delivery', 'total_price', 'total_freight', 'total_order_value', 'item_count', 'product_category_english', 'review_score', 'sentiment_label']

🔍 Preview:
                           order_id                customer_unique_id customer_city customer_state order_purchase_timestamp  order_year  order_month  order_dayofweek  delivery_time_days  delivery_delay_days  is_late_delivery  total_price  total_freight  total_order_value  item_count product_category_english  review_score sentiment_label
0  e481f51cbdc54678b7cc49136f2d6af7  7c396fd4830fd04220f754e42b4e5bff     sao paulo             SP      2017-10-02 10:56:33        2017           10                0                   8                   -8      

In [7]:
# ============================================
# CELL 6: SIMPAN KE FOLDER FINAL
# ============================================
print("💾 Menyimpan ke folder final...\n")

# Master dataset (untuk BI dashboard)
master_final.to_csv(
    FINAL_PATH / 'master_dataset.csv',
    index=False
)
print(f"✅ master_dataset.csv    → {len(master_final):,} rows")

# RFM table (untuk AI Customer Segmentation)
rfm.to_csv(
    FINAL_PATH / 'rfm_table.csv',
    index=False
)
print(f"✅ rfm_table.csv         → {len(rfm):,} rows")

# Reviews with text (untuk AI Sentiment Analysis)
reviews_with_text = pd.read_csv(PROCESSED_PATH / 'reviews_with_text.csv')
reviews_with_text.to_csv(
    FINAL_PATH / 'sentiment_dataset.csv',
    index=False
)
print(f"✅ sentiment_dataset.csv → {len(reviews_with_text):,} rows")

print(f"\n🎉 Notebook 03 selesai!")
print(f"➡️  Langkah berikutnya: 04_customer_segmentation.ipynb")
print(f"\n📁 File final tersimpan di: {FINAL_PATH}")
print(f"\n📊 Summary:")
print(f"   master_dataset   → untuk Dashboard BI")
print(f"   rfm_table        → untuk AI Customer Segmentation")
print(f"   sentiment_dataset → untuk AI Sentiment Analysis")

💾 Menyimpan ke folder final...

✅ master_dataset.csv    → 96,470 rows
✅ rfm_table.csv         → 93,350 rows
✅ sentiment_dataset.csv → 40,783 rows

🎉 Notebook 03 selesai!
➡️  Langkah berikutnya: 04_customer_segmentation.ipynb

📁 File final tersimpan di: D:\CAPSTONE\data\final

📊 Summary:
   master_dataset   → untuk Dashboard BI
   rfm_table        → untuk AI Customer Segmentation
   sentiment_dataset → untuk AI Sentiment Analysis
